# 1 · Build the corpus

For each chosen year: find every candidate paper (the facility's institutional repositories
and Europe PMC), fetch its full text, screen it with the rules, and embed it for the check list.
Everything is stored in Google Drive (`DATA`) and is **resumable**: an interrupted run continues
where it stopped.

Run once per year (and again the following spring, when repositories have caught up).
Then use notebook **2 · Find and validate**.

**Time per year**: ~40-60 min to fetch and screen; embedding ~30-40 min on the free CPU runtime,
a few minutes on a GPU (Runtime → Change runtime type → T4 GPU; picked up automatically).

## Settings

In [ ]:
FACILITY = 'aic-turku'                    # folder under facilities/ in the repository
YEARS = [2024, 2023, 2022]                # years to build
DATA = '/content/drive/MyDrive/AIC-publications'   # working data in your Drive (private)
REPO = 'https://github.com/AIC-Turku/facility-publication-finder'

## Setup

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DATA, exist_ok=True)

# The code is cloned fresh into the temporary Colab disk every session; all data stays in DATA
# (Google Drive), so updating the code never touches results.
CODE = '/content/code'                    # temporary disk; rules and confirmed papers come with it
if os.path.exists(CODE):
    subprocess.run(['git', '-C', CODE, 'pull', '--ff-only', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, CODE], check=True)
%pip install -q "{CODE}[embed,openiris]"
os.environ['PUBS_DATA'] = DATA                                  # every command reads and writes data here
os.environ['PUBS_FACILITY'] = f'{CODE}/facilities/{FACILITY}'   # rules + confirmed papers
!facility-pubs check-facility

## Optional private inputs

Bookings and the staff list only add a private review score; they never change what is
validated or published.

In [ ]:
UPLOAD_PRIVATE = False   # OpenIRIS admin export (.xlsx) and/or staff.csv ("Lastname, Firstname")

import shutil
PRIVATE = '/content/private'              # temporary disk: never in Drive or the repository
os.makedirs(PRIVATE, exist_ok=True)
os.environ['PUBS_PRIVATE'] = PRIVATE      # private inputs and everything derived from them stay here
if UPLOAD_PRIVATE:
    from google.colab import files
    for name in files.upload():
        shutil.move(name, os.path.join(PRIVATE, name))
    for x in [f for f in os.listdir(PRIVATE) if f.endswith('.xlsx')]:
        !facility-pubs import-openiris --xlsx "{PRIVATE}/{x}"
print('private inputs:', sorted(os.listdir(PRIVATE)) or 'none')

## Fetch, screen and embed

In [ ]:
for y in YEARS:
    print(f'=================== {y}')
    !facility-pubs sweep --year {y}
# every swept year in DATA takes part in the ranking (more years = better scoring)
from facility_pubs.workflow import swept_years
SWEPT_ARG = ' '.join(map(str, swept_years()))
!facility-pubs embed --years {SWEPT_ARG}

Done. Open [notebook 2 · Find and validate](https://colab.research.google.com/github/AIC-Turku/facility-publication-finder/blob/main/notebooks/2_find_and_validate.ipynb)
to create the validation Sheet for a year. Switching between a GPU and a CPU runtime is fine:
the embeddings already made are reused.